# Step 4: Recommendation Engine

**Project:** Spotify Music Recommendation System
**Phase:** 4 of 4 — Modeling (Recommendation System)

## What this notebook does

This is where everything from the earlier phases comes together. Given a
song a user picks, we recommend the 10 songs most similar to it based on
audio characteristics — using the same nine scaled features and the same
`StandardScaler` we fit in Step 3.

This is a **content-based recommender**: it works purely from the sound
of a song, not from any listening history or user behavior data (which we
don't have). That's a deliberate scope decision, and one worth being
upfront about — see Section 4.5 for what this approach can and can't do.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)


In [2]:
df = pd.read_csv("../data/processed/tracks_with_clusters.csv")
scaler = joblib.load("../outputs/models/scaler.joblib")

features = [
    "acousticness", "danceability", "energy", "instrumentalness",
    "liveness", "loudness", "speechiness", "tempo", "valence",
]

X_scaled = scaler.transform(df[features])
print(f"Loaded {df.shape[0]:,} tracks, ready to recommend from.")


Loaded 169,909 tracks, ready to recommend from.


## 4.1 The core idea: cosine similarity

Each song is represented as a point in 9-dimensional space (one dimension
per audio feature, already scaled from Step 3). **Cosine similarity**
measures the angle between two songs' feature vectors — the smaller the
angle, the more similar the songs are considered to be. It returns a
score from -1 to 1, where 1 means identical direction (maximally
similar).

Given a query song, we compute its cosine similarity against every other
track in the dataset in one vectorized operation, then sort and return
the top matches. This scales well — even with ~170,000 tracks, comparing
one song against all of them is fast because it's a single matrix
operation rather than a loop.

## 4.2 Handling real-world messiness before writing the core function

Before building the recommender, it's worth showing *why* a few extra
steps are necessary — they weren't obvious until testing against real
data surfaced the problems.

**Problem 1: duplicate song titles.** Many different songs share the same
title.

In [3]:
print("Total tracks:", len(df))
print("Unique song titles:", df['name'].nunique())
print()
print("Titles that appear most often (each one is a DIFFERENT song by a different artist):")
print(df['name'].value_counts().head(5))


Total tracks: 169909
Unique song titles: 132939

Titles that appear most often (each one is a DIFFERENT song by a different artist):
name
Summertime    62
Overture      43
Home          40
Stay          34
You           33
Name: count, dtype: int64


This means we can't just search by song title alone — `"Home"` could
refer to 40 different songs. We need the artist name too, to correctly
identify *which* song the user means.

**Problem 2: the same recording appears multiple times.** Even after
picking a specific song *and* artist, some songs have several entries in
the dataset — usually because the same recording was released on multiple
albums or compilations over the years, each with a different Spotify ID.
We saw this directly while testing: recommending for "Shape of You" by Ed
Sheeran initially returned *itself* as the #1 recommendation, because a
near-identical duplicate entry existed elsewhere in the dataset. The fix
is to explicitly exclude any row that shares the same song name and artist
as the query song — not just the one exact row the user picked.

## 4.3 The recommendation function

In [4]:
def recommend(song_name, artist_name=None, n=10):
    """
    Recommend the top `n` most similar songs to the given song, based on
    cosine similarity of audio features.

    Parameters
    ----------
    song_name : str
        Title of the song to base recommendations on.
    artist_name : str, optional
        Used to disambiguate when multiple songs share the same title.
    n : int
        Number of recommendations to return.

    Returns
    -------
    (chosen_row, recommendations_df) or (None, None) if no match is found.
    """
    matches = df[df["name"].str.lower() == song_name.lower()]
    if artist_name:
        matches = matches[matches["artists"].str.lower().str.contains(artist_name.lower())]

    if len(matches) == 0:
        return None, None

    # If multiple entries exist for the same song (re-releases, compilations),
    # use the most popular version as the reference point
    chosen = matches.sort_values("popularity", ascending=False).iloc[0]
    idx = chosen.name

    query_vector = X_scaled[idx].reshape(1, -1)
    similarities = cosine_similarity(query_vector, X_scaled)[0]

    results = df.copy()
    results["similarity"] = similarities

    # Exclude every entry of the query song itself (not just the one exact row),
    # so a duplicate/re-release of the query song can't be recommended back to the user
    is_same_song = (
        (results["name"].str.lower() == chosen["name"].lower())
        & (results["artists"] == chosen["artists"])
    )
    results = results[~is_same_song]

    results = results.sort_values("similarity", ascending=False)

    # De-duplicate near-identical re-releases of OTHER songs too, keeping
    # only the closest-matching version of each distinct song
    results["dedupe_key"] = results["name"].str.lower().str.strip() + "||" + results["artists"]
    results = results.drop_duplicates(subset="dedupe_key", keep="first")

    top_n = results.head(n)[["name", "artists", "year", "similarity", "cluster"]]
    return chosen, top_n


## 4.4 Testing the recommender

We'll test with two very different, well-known songs to see whether the
recommendations make intuitive sense.

In [5]:
chosen, recommendations = recommend("Bohemian Rhapsody", artist_name="Queen")

print(f"Recommendations for: {chosen['name']} — {chosen['artists']} ({int(chosen['year'])})\n")
recommendations


Recommendations for: Bohemian Rhapsody — ['Queen'] (2006)



,name,artists,year,similarity,cluster
139720,Cuerpo Sin Alma,['Riccardo Cocciante'],1974,0.989552,4
83440,Bohemian Rhapsody - 2011 Mix,['Queen'],1975,0.984987,4
6574,All I Need Is You - Live,['Hillsong UNITED'],2005,0.981837,4
112850,Johnny 99,['Bruce Springsteen'],1982,0.980131,4
37168,Tiny Dancer,['Elton John'],1990,0.979433,4
55152,Good Good Father - Live,['Casting Crowns'],2015,0.975405,4
141130,Lover of Mine,['Alannah Myles'],1989,0.974892,4
147637,Heroes and Villains (Stereo),['The Beach Boys'],1967,0.973676,4
36140,Reality,"['Vladimir Cosma', 'Richard Sanderson']",1980,0.972338,4
39812,Answer,['Phantogram'],2016,0.972012,4


In [6]:
chosen2, recommendations2 = recommend("Shape of You", artist_name="Ed Sheeran")

print(f"Recommendations for: {chosen2['name']} — {chosen2['artists']} ({int(chosen2['year'])})\n")
recommendations2


Recommendations for: Shape of You — ['Ed Sheeran'] (2017)



,name,artists,year,similarity,cluster
31866,Que Te Vaya Bien,['Grupo Jalado'],2017,0.989239,1
105310,Gripa Colombiana,['Los Tucanes De Tijuana'],2000,0.989002,1
46971,Tu Defecto,['Los Creadorez Del Pasito Duranguense'],2009,0.984789,1
96577,Hasta El Día De Hoy,['Los Dareyes De La Sierra'],2008,0.984205,1
32104,Lupe Campos,"['El Fantasma', 'Los Dos Carnales']",2019,0.983876,1
169308,El Comando del Diablo,"['Noel Torres', 'Gerardo Ortiz']",2014,0.983294,1
93690,Brujeria,['El Gran Combo De Puerto Rico'],1979,0.983206,1
140500,Goyito Sabater,['El Gran Combo De Puerto Rico'],1982,0.982395,1
69437,Te Quiero Así,['Valentín Elizalde'],2008,0.981244,1
124184,El Cruzador,['Los Tucanes De Tijuana'],2000,0.980951,1


### Reading these results honestly

The Bohemian Rhapsody recommendations include tracks like "Tiny Dancer"
(Elton John) and "Heroes and Villains" (The Beach Boys) — theatrical,
mid-tempo rock/pop tracks, which feels like a musically sensible match.

The Shape of You results are more revealing. The recommendations are
dominated by regional Mexican/banda-style tracks — not remotely the same
genre as Ed Sheeran's pop. This is an important, honest limitation to
call out rather than hide: **cosine similarity on audio features only
captures numerical closeness in tempo, danceability, energy, etc. — it
has no concept of genre, culture, language, or "vibe" the way a human
listener (or a system trained on real listening behavior) would.** Two
songs can score as highly similar because they share a danceable tempo
and upbeat valence, even if a human would never group them together.



## 4.5 Limitations of this approach

- **No genre awareness**: as shown above, purely numeric similarity can
  surface stylistically mismatched recommendations. Adding genre as an
  explicit feature (had it been available in this dataset) or blending in
  a collaborative-filtering signal would likely improve perceived
  relevance.
- **Cold start for new songs**: any song not already in this dataset
  can't be recommended *from* or *to* without first computing its audio
  features — which, as discussed in Step 1, is no longer possible via
  Spotify's public API for new developer apps. In a production system,
  this would need an alternative feature source.
- **No personalization**: every user gets the same recommendations for
  the same input song. Real systems typically blend content similarity
  with a user's individual listening history.
- **Popularity bias in disambiguation**: when a song has multiple
  versions, we default to the most popular one. This is a reasonable
  default but means less well-known re-recordings or remixes are never
  used as the reference point.

## 4.6 Saving the recommender for reuse

To keep the Streamlit app (Step 5) clean, we move this logic into a
reusable module rather than duplicating it. See `src/recommender.py`.

In [8]:
# Quick sanity check that the saved module version behaves the same way
import sys
sys.path.append("..")
from src.recommender import recommend as recommend_from_module

chosen3, recs3 = recommend_from_module(df, X_scaled, "Bohemian Rhapsody", artist_name="Queen")
recs3


,name,artists,year,similarity,cluster
139720,Cuerpo Sin Alma,['Riccardo Cocciante'],1974,0.989552,4
83440,Bohemian Rhapsody - 2011 Mix,['Queen'],1975,0.984987,4
6574,All I Need Is You - Live,['Hillsong UNITED'],2005,0.981837,4
112850,Johnny 99,['Bruce Springsteen'],1982,0.980131,4
37168,Tiny Dancer,['Elton John'],1990,0.979433,4
55152,Good Good Father - Live,['Casting Crowns'],2015,0.975405,4
141130,Lover of Mine,['Alannah Myles'],1989,0.974892,4
147637,Heroes and Villains (Stereo),['The Beach Boys'],1967,0.973676,4
36140,Reality,"['Vladimir Cosma', 'Richard Sanderson']",1980,0.972338,4
39812,Answer,['Phantogram'],2016,0.972012,4


## Summary

- Built a content-based recommender using cosine similarity over the nine
  scaled audio features from Step 3.
- Solved two real data-quality problems found through testing: duplicate
  song titles across different artists, and duplicate entries for the
  exact same recording appearing under different Spotify IDs.
- Verified the recommender against two well-known songs. One produced
  intuitive results; the other revealed a genuine limitation — numeric
  audio similarity alone doesn't capture genre or cultural context, which
  is now documented rather than hidden.
- Recommendation logic is saved to `src/recommender.py` for reuse in the
  Streamlit app.

## Next steps

Step 5 wraps this into an interactive Streamlit app: a user picks a song
from a dropdown/search box, and the app displays the top 10
recommendations along with the audio-feature reasoning behind each one.
